In [ ]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pingouin as pg
import matplotlib
from glob import glob
import sys
sys.path.extend(['/Users/amonast/Documents/GitHub/zlab_miniscope'])
from load_traces import load_aligned_act,build_intervals
from scipy.ndimage import gaussian_filter1d
from mpl_toolkits.mplot3d import Axes3D
# mixed color
matplotlib.rcParams['svg.fonttype']='none'
traces_dir = '/Users/amonast/Desktop/miniscope/Traces'
behavior_csv = '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv'
data_info_csv = '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv'

In [ ]:
import sys
sys.path.extend(['/Users/amonast/Desktop/miniscope'])
from utils import ridge_plot

color_map_categories = {
    'Aggressive':        "#fe0806",
    'Pro-Social':        "#55a0fb",
    'Social Avoidance':  '#b856d7',
    'Asocial':           '#e7e6e6',}
mixed_color = "#19893B"
color_map = color_map_categories

def label_bouts(act):
    # detect every transition (including NaN→behavior and behavior→NaN)
    act['bout_id'] = (act['behavior_type'].fillna('_none_')
                        .ne(act['behavior_type'].fillna('_none_').shift())
                        .cumsum())

    # per-behavior_type bout number (1, 2, 3 …)
    act['bout_num'] = act.groupby('behavior_type', dropna=False)['bout_id'] \
                        .transform(lambda x: pd.factorize(x)[0] + 1)
    return act

def plot_tuned_cells(act, intervals, tuned, animal, session, behavior_type,
                     cell_subset=None, color_map=None, baseline_cutoff=120,
                     figsize=(9, 5), save_path=None):
    """
    Plot behavior raster above ridge plot of cells tuned to one behavior_type.

    Parameters
    ----------
    cell_subset : list, optional
        If provided, plot these cell IDs instead of all tuned cells for the behavior_type.
    """
    if cell_subset is not None:
        cell_ids = sorted(cell_subset)
    else:
        mask = ((tuned["animal"] == animal) &
                (tuned["session"] == session) &
                (tuned["behavior_type"] == behavior_type))
        rows = tuned.loc[mask]
        if rows.empty or not rows["tuned_cells"].iloc[0]:
            print(f"No significantly tuned cells for {animal} {session} {behavior_type}")
            return
        cell_ids = sorted(rows["tuned_cells"].iloc[0])

    sub = act.loc[(animal, session)]
    times = sub.index.total_seconds().to_numpy()
    signal = sub[cell_ids].to_numpy().T  # (n_cells, T)

    color = color_map.get(behavior_type, "#333333") if color_map else "#333333"

    ani_ivs = intervals[
        (intervals["animal"]        == animal) &
        (intervals["session"]       == session) &
        (intervals["behavior_type"] == behavior_type)
    ]

    fig, (ax_beh, ax_ridge) = plt.subplots(
        2, 1, figsize=figsize, sharex=True,
        gridspec_kw={"height_ratios": [1, 6], "hspace": 0.05}
    )

    for _, row in ani_ivs.iterrows():
        ax_beh.barh(y=0, width=row["end"] - row["start"], left=row["start"],
                    height=0.8, color=color, alpha=0.8)

    ax_beh.axvline(baseline_cutoff, color="r", linestyle="--", linewidth=1)
    ax_beh.set_ylim([-0.5, 0.5])
    ax_beh.set_yticks([])
    ax_beh.set_ylabel("Behavior", fontsize=8, rotation=0, labelpad=45, va="center")
    ax_beh.spines[["top", "right", "left"]].set_visible(False)

    ridge_plot(signal, times, color=color, ax=ax_ridge,
               alpha=0.8, ytick_spacing=3, line_width=1, trace_spacing=4)

    ax_ridge.axvline(baseline_cutoff, color="r", linestyle="--", linewidth=1)
    ax_ridge.set_xlabel("Time (s)")
    plt.suptitle(f"{animal} {session} — {behavior_type} — {len(cell_ids)} cells")

    plt.tight_layout()
    plt.show()

    if save_path:
        fig.savefig(save_path, transparent=True)

def plot_tuned_cells_shaded(act, intervals, tuned, animal, session, behavior_type,
                            cell_subset=None, color_map=None, baseline_cutoff=120,
                            figsize=(9, 5), save_path=None):
    """
    Like plot_tuned_cells but overlays shaded spans for behavior intervals
    directly on the ridge plot instead of a separate raster subplot.
    """
    if cell_subset is not None:
        cell_ids = sorted(cell_subset)
    else:
        mask = ((tuned['animal'] == animal) &
                (tuned['session'] == session) &
                (tuned['behavior_type'] == behavior_type))
        rows = tuned.loc[mask]
        if rows.empty or not rows['tuned_cells'].iloc[0]:
            print(f'No significantly tuned cells for {animal} {session} {behavior_type}')
            return
        cell_ids = sorted(rows['tuned_cells'].iloc[0])

    sub = act.loc[(animal, session)]
    times = sub.index.total_seconds().to_numpy()
    signal = sub[cell_ids].to_numpy().T

    color = color_map.get(behavior_type, '#333333') if color_map else '#333333'

    ani_ivs = intervals[
        (intervals['animal']        == animal) &
        (intervals['session']       == session) &
        (intervals['behavior_type'] == behavior_type)
    ]

    fig, ax = plt.subplots(figsize=figsize)

    ridge_plot(signal, times, color='#333333', ax=ax,
               alpha=0.8, ytick_spacing=3, line_width=1, trace_spacing=4)

    for _, row in ani_ivs.iterrows():
        ax.axvspan(row['start'], row['end'], color=color, alpha=0.3, linewidth=0)

    ax.axvline(baseline_cutoff, color='r', linestyle='--', linewidth=1)
    ax.set_xlabel('Time (s)')
    plt.title(f'{animal} {session} — {behavior_type} — {len(cell_ids)} cells')
    plt.tight_layout()
    plt.show()

    if save_path:
        fig.savefig(save_path, transparent=True)


def get_exclusive_tuned(tuned, animal, session):
    """
    Return a dict {behavior_type: [cell_ids]} containing only cells
    tuned to exactly one behavior type (not mixed selective).
    """
    sub = tuned[(tuned["animal"] == animal) & (tuned["session"] == session)]

    cell_btype = {}
    for _, row in sub.iterrows():
        for c in row["tuned_cells"]:
            cell_btype.setdefault(c, []).append(row["behavior_type"])

    exclusive = {c: btypes[0] for c, btypes in cell_btype.items() if len(btypes) == 1}

    result = {}
    for _, row in sub.iterrows():
        bt = row["behavior_type"]
        result[bt] = [c for c in row["tuned_cells"] if exclusive.get(c) == bt]

    return result

def get_mixed_tuned(tuned, animal, session):
    """
    Return a dict of mixed-selective cells for one animal/session.

    Keys are frozensets of behavior_type pairs/triples, values are cell ID lists.
    """
    sub = tuned[(tuned["animal"] == animal) & (tuned["session"] == session)]

    cell_btype = {}
    for _, row in sub.iterrows():
        for c in row["tuned_cells"]:
            cell_btype.setdefault(c, set()).add(row["behavior_type"])

    result = {}
    for c, btypes in cell_btype.items():
        if len(btypes) > 1:
            key = frozenset(btypes)
            result.setdefault(key, []).append(c)

    return result

def get_zscores_for_tuned(tuned_dict, zscores, animal, session):
    """Return {behavior_type: Series of z-scores} for exclusively tuned cells."""
    result = {}
    for bt, cell_ids in tuned_dict.items():
        if not cell_ids:
            continue
        row = zscores[
            (zscores['animal'] == animal) &
            (zscores['session'] == session) &
            (zscores['behavior_type'] == bt)
        ]
        result[bt] = row[cell_ids].iloc[0]
    return result


def plot_all_tuned(act, intervals, tuned, animal, session,
                   btypes=None, color_map=None, baseline_cutoff=120,
                   positive_only=False, figsize=(12, 7), save_path=None):
    """
    Single figure with behavior raster on top and ridge plot below,
    showing exclusively tuned cells for each behavior type chunked and colored.
    """
    if btypes is None:
        btypes = CATEGORIES
    if color_map is None:
        color_map = {}

    exclusive = get_exclusive_tuned(tuned, animal, session)

    if positive_only:
        z = get_zscores_for_tuned(exclusive, zscores, animal, session)
        exclusive = {bt: [c for c in cells if z.get(bt, pd.Series(dtype=float)).get(c, 0) > 0]
                     for bt, cells in exclusive.items()}

    cell_ids = []
    colors   = []
    for bt in btypes:
        cells = exclusive.get(bt, [])
        cell_ids.extend(cells)
        colors.extend([color_map.get(bt, '#333333')] * len(cells))

    if not cell_ids:
        print(f'No exclusively tuned cells for {animal} {session}')
        return

    sub    = act.loc[(animal, session)]
    times  = sub.index.total_seconds().to_numpy()
    signal = sub[cell_ids].to_numpy().T

    ani_ivs = intervals[
        (intervals['animal']        == animal) &
        (intervals['session']       == session) &
        (intervals['behavior_type'].isin(btypes))
    ]

    fig, (ax_beh, ax_ridge) = plt.subplots(
        2, 1, figsize=figsize, sharex=True,
        gridspec_kw={'height_ratios': [1, 6], 'hspace': 0.05}
    )

    seen = set()
    for _, row in ani_ivs.iterrows():
        bt = row['behavior_type']
        ax_beh.barh(y=0, width=row['end'] - row['start'], left=row['start'],
                    height=0.8, color=color_map.get(bt, '#aaaaaa'),
                    label=bt if bt not in seen else '', alpha=0.8)
        seen.add(bt)

    ax_beh.axvline(baseline_cutoff, color='r', linestyle='--', linewidth=1)
    ax_beh.set_ylim([-0.5, 0.5])
    ax_beh.set_yticks([])
    ax_beh.set_ylabel('Behavior', fontsize=8, rotation=0, labelpad=45, va='center')
    ax_beh.spines[['top', 'right', 'left']].set_visible(False)
    ax_beh.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

    ridge_plot(signal, times, color=colors, ax=ax_ridge,
               alpha=0.8, ytick_spacing=3, line_width=1, trace_spacing=4)

    ax_ridge.axvline(baseline_cutoff, color='r', linestyle='--', linewidth=1)
    ax_ridge.set_xlabel('Time (s)')

    handles = [
        plt.Line2D([0], [0], color=color_map.get(bt, '#333333'), linewidth=2,
                   label=f'{bt} (n={len(exclusive.get(bt, []))})')
        for bt in btypes
    ]
    ax_ridge.legend(handles=handles, frameon=False,
                    bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.suptitle(f'{animal} {session} — {len(cell_ids)} exclusively tuned cells')
    plt.tight_layout()
    plt.show()

    if save_path:
        fig.savefig(save_path, transparent=True)


def get_first_prosocial_window(intervals: pd.DataFrame,
                               act: pd.DataFrame,
                               window_sec: float,
                               pre_sec: float = 0.0) -> tuple[np.ndarray, np.ndarray] | tuple[None, None]:
    """
    Find the first Pro-Social interval onset, extract pre_sec before and
    window_sec after onset. Returns (t_rel, T_window x N_cells) or (None, None).
    """
    prosocial = intervals[intervals["behavior_type"] == "Pro-Social"].sort_values("start")
    if prosocial.empty:
        return None, None

    onset   = prosocial.iloc[0]["start"]
    t_start = onset - pre_sec
    t_end   = onset + window_sec

    cell_cols  = [c for c in act.columns if str(c).isdigit() or isinstance(c, (int, np.integer))]
    window_act = act[(act.index >= t_start) & (act.index < t_end)][cell_cols]

    t_rel = window_act.index.to_numpy() - onset
    return t_rel, window_act.values


def _mixed_cell_color(c, combo, animal, session, color_map, mixed_color):
    """
    Color a mixed cell by how many of its tuned behaviors have z > 0.
    - Exactly one z > 0  → that behavior's color
    - All others (0 or >1 positive) → mixed_color
    """
    pos_btypes = []
    for bt in combo:
        z_row = zscores[
            (zscores['animal'] == animal) &
            (zscores['session'] == session) &
            (zscores['behavior_type'] == bt)
        ]
        if not z_row.empty and c in z_row.columns:
            if float(z_row[c].iloc[0]) > 0:
                pos_btypes.append(bt)
    return color_map.get(pos_btypes[0], mixed_color) if len(pos_btypes) == 1 else mixed_color


def plot_all_tuned_with_mixed(act, intervals, tuned, animal, session,
                               btypes=None, color_map=None, mixed_color='#9B59B6',
                               baseline_cutoff=120, positive_only=False,
                               figsize=(12, 7), save_path=None):
    """
    Ridge plot of all significantly tuned cells including mixed-selective ones.
    Exclusively tuned cells are chunked by behavior type; mixed cells follow at the bottom.
    Mixed cells are colored mixed_color when z > 0 for multiple behaviors,
    or the single positive behavior's color when only one behavior has z > 0.
    """
    if btypes is None:
        btypes = CATEGORIES
    if color_map is None:
        color_map = {}

    exclusive = get_exclusive_tuned(tuned, animal, session)
    mixed     = get_mixed_tuned(tuned, animal, session)

    if positive_only:
        z = get_zscores_for_tuned(exclusive, zscores, animal, session)
        exclusive = {bt: [c for c in cells if z.get(bt, pd.Series(dtype=float)).get(c, 0) > 0]
                     for bt, cells in exclusive.items()}

    cell_ids = []
    colors   = []

    # Exclusive chunks
    for bt in btypes:
        cells = exclusive.get(bt, [])
        cell_ids.extend(cells)
        colors.extend([color_map.get(bt, '#333333')] * len(cells))

    # Mixed chunk — color per cell based on how many behaviors have z > 0
    n_truly_mixed = 0
    for combo, cells in mixed.items():
        if not combo.issubset(set(btypes)):
            continue
        for c in cells:
            c_color = _mixed_cell_color(c, combo, animal, session, color_map, mixed_color)
            if c_color == mixed_color:
                n_truly_mixed += 1
            cell_ids.append(c)
            colors.append(c_color)

    if not cell_ids:
        print(f'No tuned cells for {animal} {session}')
        return

    sub    = act.loc[(animal, session)]
    times  = sub.index.total_seconds().to_numpy()
    signal = sub[cell_ids].to_numpy().T

    ani_ivs = intervals[
        (intervals['animal']        == animal) &
        (intervals['session']       == session) &
        (intervals['behavior_type'].isin(btypes))
    ]

    fig, (ax_beh, ax_ridge) = plt.subplots(
        2, 1, figsize=figsize, sharex=True,
        gridspec_kw={'height_ratios': [1, 6], 'hspace': 0.05}
    )

    seen = set()
    for _, row in ani_ivs.iterrows():
        bt = row['behavior_type']
        ax_beh.barh(y=0, width=row['end'] - row['start'], left=row['start'],
                    height=0.8, color=color_map.get(bt, '#aaaaaa'),
                    label=bt if bt not in seen else '', alpha=0.8)
        seen.add(bt)

    ax_beh.axvline(baseline_cutoff, color='r', linestyle='--', linewidth=1)
    ax_beh.set_ylim([-0.5, 0.5])
    ax_beh.set_yticks([])
    ax_beh.set_ylabel('Behavior', fontsize=8, rotation=0, labelpad=45, va='center')
    ax_beh.spines[['top', 'right', 'left']].set_visible(False)
    ax_beh.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

    ridge_plot(signal, times, color=colors, ax=ax_ridge,
               alpha=0.8, ytick_spacing=3, line_width=1, trace_spacing=4)

    ax_ridge.axvline(baseline_cutoff, color='r', linestyle='--', linewidth=1)
    ax_ridge.set_xlabel('Time (s)')

    n_mixed_total = sum(len(c) for combo, c in mixed.items() if combo.issubset(set(btypes)))
    handles = [
        plt.Line2D([0], [0], color=color_map.get(bt, '#333333'), linewidth=2,
                   label=f'{bt} (n={len(exclusive.get(bt, []))})')
        for bt in btypes
    ] + [plt.Line2D([0], [0], color=mixed_color, linewidth=2,
                    label=f'Mixed (n={n_mixed_total}, {n_truly_mixed} both+)')]
    ax_ridge.legend(handles=handles, frameon=False,
                    bbox_to_anchor=(1.01, 1), loc='upper left')

    plt.suptitle(f'{animal} {session} — {len(cell_ids)} tuned cells (incl. mixed)')
    plt.tight_layout()
    plt.show()

    if save_path:
        fig.savefig(save_path, transparent=True)

# Get Tuned Data

In [ ]:
CATEGORIES = ['Aggressive', 'Pro-Social', 'Social Avoidance']

data_dir = '/Users/amonast/Desktop/miniscope/data'
pval_files = glob(f"{data_dir}/tuning/p_values*.csv")
rows = []
for f in pval_files:
    df1 = pd.read_csv(f)
    rows.append(df1)

df = pd.concat(rows)


cell_cols = [c for c in df.columns if c not in ('animal', 'session', 'behavior_type')]

sig = (df.set_index(['animal', 'session', 'behavior_type'])[cell_cols].le(0.05))

tuned = (sig.apply(lambda row: row.index[row].tolist(), axis=1)
            .rename('tuned_cells')
            .reset_index())
## exlcude these animals cells as aggessive tuned b/c one was the intruder mouse and one only had one tiny bout 
tuned = tuned[~(
    (tuned["animal"].isin(["C2M4", "C2F1"])) &
    (tuned["behavior_type"] == "Aggressive")
)]
tuned

In [ ]:
zscore_files = glob(f"{data_dir}/tuning/z_scores*.csv")
zrows = [pd.read_csv(f) for f in zscore_files]
zdf = pd.concat(zrows).drop_duplicates(subset=['animal', 'session', 'behavior_type'])
zdf = zdf[~((zdf["animal"].isin(["C2M4", "C2F1"])) &
    (zdf["behavior_type"] == "Aggressive"))]
zcell_cols = [c for c in zdf.columns if c not in ('animal', 'session', 'behavior_type')]
zscores = tuned[['animal', 'session', 'behavior_type']].merge(
    zdf[['animal', 'session', 'behavior_type'] + zcell_cols],
    on=['animal', 'session', 'behavior_type'],
    how='left'
)



In [ ]:
tuned_beh = tuned[tuned['behavior_type'] != 'Other']

# ── total cells recorded per animal/session (non-NaN p-values) ───────────────
n_all = (df.drop_duplicates(subset=['animal', 'session', 'behavior_type'])
           .set_index(['animal', 'session', 'behavior_type'])[cell_cols]
           .notna().sum(axis=1)
           .groupby(['animal', 'session']).max()      # same cells in every behavior row
           .rename('n_allcells'))

# ── unique tuned cells per animal/session (union across behaviors) ───────────
n_tuned_total = (tuned_beh.groupby(['animal', 'session'])['tuned_cells']
                      .apply(lambda lists: len(set().union(*lists)))
                      .rename('n_total_tuned'))

prop = pd.concat([n_tuned_total, n_all], axis=1).fillna({'n_total_tuned': 0})
prop['prop_tuned'] = prop['n_total_tuned'] / prop['n_allcells']

# ── per animal (pool sessions) ───────────────────────────────────────────────
per_animal = prop.groupby('animal')[['n_total_tuned', 'n_allcells']].sum()
per_animal['prop_tuned'] = per_animal['n_total_tuned'] / per_animal['n_allcells']

# ── overall ──────────────────────────────────────────────────────────────────
overall = per_animal['n_total_tuned'].sum() / per_animal['n_allcells'].sum()
print(f"Overall: {overall:.3f}  |  mean ± sem across animals: "
      f"{per_animal['prop_tuned'].mean():.3f} ± {per_animal['prop_tuned'].sem():.3f}")
per_animal

In [ ]:
zscores

In [ ]:
# ── 4-category color map ──────────────────────────────────────────────────────
color_map_categories = {
    'Aggressive':        "#fe0806",
    'Pro-Social':        "#55a0fb",
    'Social Avoidance':            '#b856d7',
    'Asocial':             '#e7e6e6',}

mixed_color = "#19893B"

# Example plot individual ridgeplots 

In [ ]:
animal = 'C2F1'
session ='R1'
cell_cols = [c for c in df.columns if c not in ('animal', 'session', 'behavior_type')]
intervals = build_intervals(animal, session)
act = load_aligned_act(animal,session,
                       '/Users/amonast/Desktop/miniscope/Traces',
                       '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv',
                       '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv',)

cell_cols = [c for c in act.columns if str(c).isdigit()]
act[cell_cols] = gaussian_filter1d(act[cell_cols].to_numpy(), sigma=4, axis=0)

In [ ]:
plot_tuned_cells(act, intervals, tuned, animal, session, "Pro-Social", 
                 color_map=color_map_categories,figsize=(12,6))


In [ ]:
plot_tuned_cells(act, intervals, tuned, animal, session, "Aggressive", color_map=color_map_categories,figsize=(12,6))


In [ ]:
plot_tuned_cells(act, intervals, tuned, animal, session, "Social Avoidance", color_map=color_map_categories,figsize=(12,6))


# All cells from all animals - what proportion are pro social versus anti social?

In [ ]:
tuned["n_tuned"] = tuned["tuned_cells"].apply(len)

# ── Which animals displayed aggression (any session) ─────────────────────────
agg_animals = set(tuned.loc[(tuned["behavior_type"] == "Aggressive") 
                       & (tuned["n_tuned"] > 0), "animal"])
non_agg_animals = set(tuned["animal"]) - agg_animals
# -- dont include C2M4
agg_animals.difference_update(["C2M4", "C2F1"])
non_agg_animals.update(["C2M4", "C2F1"])

tuned["n_negative"] = tuned.apply(n_negative, axis=1)
tuned["n_positive"] = tuned.apply(n_positive, axis=1)

social_btypes = ["Pro-Social", "Social Avoidance"]
all_btypes    = ["Pro-Social", "Social Avoidance", "Aggressive"]

props_non_agg = proportion_counts(non_agg_animals, social_btypes)
props_agg     = proportion_counts(agg_animals,     all_btypes)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))

ax1.bar(props_non_agg.index, props_non_agg.values,
        color=[color_map_categories.get(b) for b in props_non_agg.index])
ax1.set_title("Non-aggressive mice", fontsize=10)
ax1.set_ylabel("Proportion of positively tuned cells")
ax1.set_ylim(0, 1)
ax1.spines[["top", "right"]].set_visible(False)

ax2.bar(props_agg.index, props_agg.values,
        color=[color_map_categories.get(b,) for b in props_agg.index])
ax2.set_title("Aggressive mice", fontsize=10)
ax2.set_ylim(0, 1)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
fig, (ax1, ax2) = plt.subplots(2,1, figsize=(8,3),dpi=150)
for ax, props, title in [(ax1, props_non_agg, "Non-aggressive mice"),
                         (ax2, props_agg,     "Aggressive mice"),]:
    colors = [color_map_categories.get(b) for b in props.index]
    ax.pie(props.values,
        labels=props.index,
        colors=colors,
        autopct='%1.0f%%',
        textprops=dict(fontsize=12),
        startangle=90,
        wedgeprops=dict(linewidth=0.5, edgecolor='white'),)
    ax.set_title(title, fontsize=12)
plt.savefig('figures/tuning_pie_chart.svg',transparent=True)

In [ ]:
def excit_inhib_proportion(animal_set, btype):
    """For one behavior type, fraction of tuned cells that are excited vs inhibited."""
    subset = tuned[tuned["animal"].isin(animal_set) & (tuned["behavior_type"] == btype)]
    n_excited  = subset["n_positive"].sum()
    n_inhibited = subset["n_negative"].sum()
    total = n_excited + n_inhibited
    if total == 0:
        return 0.0, 0.0
    return n_excited / total, n_inhibited / total



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(8, 3))

for ax, btype in zip(axes, ['Pro-Social', 'Social Avoidance']):
    exc, inh = excit_inhib_proportion(agg_animals, btype)
    ax.pie(
        [exc, inh],
        labels=['Excited', 'Inhibited'],
        colors=["#823EC6", "#A788C4"] if btype=='Social Avoidance' else ["#2F50F5", "#528DF4"],
        autopct='%1.0f%%',
        startangle=90,
        wedgeprops=dict(linewidth=0.5, edgecolor='white'),)
    ax.set_title(btype, fontsize=10)
exc, inh = excit_inhib_proportion(agg_animals, 'Aggressive')
axes[2].pie(
    [exc, inh],
    labels=['Excited', 'Inhibited'],
    colors=["#EE3C3C", "#E7B8B8"],
    autopct='%1.0f%%',
    startangle=90,
    wedgeprops=dict(linewidth=0.5, edgecolor='white'),)
axes[2].set_title('Aggressive', fontsize=10)
plt.tight_layout()
plt.savefig('figures/sub_piecharts.svg',transparent=True)

## Venn Diagram

In [ ]:
def get_unique_ids(btype, animal_set, session, direction=None):
    """
    direction: None = all tuned, 'excited' = z>0 only, 'inhibited' = z<0 only
    """
    rows = tuned[(tuned["behavior_type"] == btype) &
                 (tuned["animal"].isin(animal_set)) &
                 (tuned["session"] == session)]
    ids = set()
    for _, r in rows.iterrows():
        for c in r.tuned_cells:
            if direction is not None:
                z_row = zscores[(zscores["animal"] == r.animal) &
                                (zscores["session"] == r.session) &
                                (zscores["behavior_type"] == btype)]
                if z_row.empty:
                    continue
                z = float(z_row[c].iloc[0])
                if direction == 'excited'  and z <= 0: continue
                if direction == 'inhibited' and z >= 0: continue
            ids.add(f"{r.animal}_{r.session}_{c}")
    return ids

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, session in zip(axes, ["R1", "R2"]):
    pro_social = get_unique_ids("Pro-Social",       agg_animals, session, direction='excited')
    soc_avoid  = get_unique_ids("Social Avoidance", agg_animals, session, direction='excited')
    aggressive = get_unique_ids("Aggressive",       agg_animals, session, direction='excited')

    venn3(
        [pro_social, soc_avoid, aggressive],
        set_labels=("Pro-Social", "Social Avoidance", "Aggressive"),
        set_colors=(color_map_categories["Pro-Social"],
                    color_map_categories["Social Avoidance"],
                    color_map_categories["Aggressive"]),
        alpha=0.6, ax=ax
    )
    ax.set_title(f"All mice — {session} (excited)")

In [ ]:
all_animals = agg_animals | non_agg_animals

pro_social = (get_unique_ids("Pro-Social",       all_animals, "R1", direction='excited') |
              get_unique_ids("Pro-Social",       all_animals, "R2", direction='excited'))
soc_avoid  = (get_unique_ids("Social Avoidance", all_animals, "R1", direction='excited') |
              get_unique_ids("Social Avoidance", all_animals, "R2", direction='excited'))
aggressive = (get_unique_ids("Aggressive",       all_animals, "R1", direction='excited') |
              get_unique_ids("Aggressive",       all_animals, "R2", direction='excited'))

fig, ax = plt.subplots(figsize=(6, 6))
venn3(
    [pro_social, soc_avoid, aggressive],
    set_labels=("Pro-Social", "Social Avoidance", "Aggressive"),
    set_colors=(color_map_categories["Pro-Social"],
                color_map_categories["Social Avoidance"],
                color_map_categories["Aggressive"]),
    alpha=0.6, ax=ax
)
ax.set_title("All mice — R1 + R2 (excited)")
plt.tight_layout()


In [ ]:
all_animals = agg_animals | non_agg_animals

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, session in zip(axes, ["R1", "R2"]):
    pro_social = get_unique_ids("Pro-Social",       all_animals, session, valence='excited')
    soc_avoid  = get_unique_ids("Social Avoidance", all_animals, session, valence='excited')
    aggressive = get_unique_ids("Aggressive",       all_animals, session, valence='excited')

    venn3([pro_social, soc_avoid, aggressive],
        set_labels=("Pro-Social", "Social Avoidance", "Aggressive"),
        set_colors=(color_map_categories["Pro-Social"],
                    color_map_categories["Social Avoidance"],
                    color_map_categories["Aggressive"]),
        alpha=0.6, ax=ax
    )
    ax.set_title(f"All mice — {session} (excited)")

plt.tight_layout()


###  All aggressive animals:

In [ ]:
from matplotlib_venn import venn3


fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, session in zip(axes, ["R1", "R2"]):
    pro_social = get_unique_ids("Pro-Social",      agg_animals, session)
    soc_avoid  = get_unique_ids("Social Avoidance", agg_animals, session)
    aggressive = get_unique_ids("Aggressive",       agg_animals, session)

    venn3(
        [pro_social, soc_avoid, aggressive],
        set_labels=("Pro-Social", "Social Avoidance", "Aggressive"),
        set_colors=(color_map_categories["Pro-Social"], color_map_categories["Social Avoidance"], color_map_categories["Aggressive"]),
        alpha=0.6,
        ax=ax
    )
    ax.set_title(f"Aggressive mice — {session}")
plt.tight_layout()
plt.show()
plt.savefig('./figures/venn_diagram_aggressive_mice_by_session.svg')

In [ ]:
for a in agg_animals:
    fig,axes = plt.subplots(1, 2, figsize=(12, 6))

    for ax, session in zip(axes, ["R1", "R2"]):
        pro_social = get_unique_ids("Pro-Social",      [a], session)
        soc_avoid  = get_unique_ids("Social Avoidance", [a], session)
        aggressive = get_unique_ids("Aggressive",       [a], session)

        venn3(
            [pro_social, soc_avoid, aggressive],
            set_labels=("Pro-Social", "Social Avoidance", "Aggressive"),
            set_colors=(color_map_categories["Pro-Social"], color_map_categories["Social Avoidance"], color_map_categories["Aggressive"]),
            alpha=0.6,
            ax=ax
        )
        ax.set_title(f"{a} — {session}")

    plt.tight_layout()
    plt.savefig(f'./figures/venn_diagram_{a}_by_session.svg')

# Ridge Plot - Exclusively tuned cells

# plot example C3M2 

In [ ]:
animal = 'C2F1'
session ='R1'

tuned_dict = get_exclusive_tuned(tuned,animal,session)
mixed_dict = get_mixed_tuned(tuned,animal,session)

cell_cols = [c for c in df.columns if c not in ('animal', 'session', 'behavior_type')]
intervals = build_intervals(animal, session)
act = load_aligned_act(animal,session,
                       '/Users/amonast/Desktop/miniscope/Traces',
                       '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv',
                       '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv',)

cell_cols = [c for c in act.columns if str(c).isdigit()]
act[cell_cols] = gaussian_filter1d(act[cell_cols].to_numpy(), sigma=4, axis=0)

In [ ]:
z = get_zscores_for_tuned(tuned_dict, zscores, animal, session)
try:
    sorted_cells_agg = z["Aggressive"].sort_values(ascending=False).index.tolist()
    plot_tuned_cells(act, intervals, tuned, animal, session, "Aggressive",
                 cell_subset=sorted_cells_agg, color_map=color_map)
except:
    pass
try:
    sorted_cells_avoid= z["Social Avoidance"].sort_values(ascending=False).index.tolist()
    plot_tuned_cells(act, intervals, tuned, animal, session, "Social Avoidance",
                 cell_subset=sorted_cells_avoid, color_map=color_map)
except:
    pass
sorted_cells_social = z["Pro-Social"].sort_values(ascending=False).index.tolist()
plot_tuned_cells(act, intervals, tuned, animal, session, "Pro-Social",
                 cell_subset=sorted_cells_social, color_map=color_map)


In [ ]:
%matplotlib widget

In [ ]:
plot_tuned_cells_shaded(act,intervals,tuned,animal,session,behavior_type='Pro-Social',cell_subset=sorted_cells_social,color_map=color_map)


In [ ]:
#plot_tuned_cells_shaded(act,intervals,tuned,animal,session,behavior_type='Aggressive',cell_subset=sorted_cells_agg,color_map=color_map)


In [ ]:
plot_tuned_cells_shaded(act,intervals,tuned,animal,session,behavior_type='Social Avoidance',cell_subset=sorted_cells_avoid,color_map=color_map)


# 3d scatter plot for all cells 

In [ ]:

for (animal, session), grp in zdf.groupby(['animal', 'session']):
    fig = plt.figure(figsize=(8, 6))
    ax  = fig.add_subplot(111, projection='3d')

    grp_r = grp.set_index('behavior_type').reindex(CATEGORIES)
    cell_cols_z = [c for c in grp_r.columns if c not in ('animal', 'session')]

    x = grp_r.loc['Aggressive',    cell_cols_z].fillna(0).values.astype(float)
    y = grp_r.loc['Pro-Social',    cell_cols_z].fillna(0).values.astype(float)
    z = grp_r.loc['Social Avoidance', cell_cols_z].fillna(0).values.astype(float)

    # Significance mask per axis from tuned
    def sig_mask(bt):
        row = tuned[(tuned['animal'] == animal) & (tuned['session'] == session)
                    & (tuned['behavior_type'] == bt)]
        if row.empty:
            return set()
        return set(row['tuned_cells'].iloc[0])

    sig_agg    = sig_mask('Aggressive')
    sig_pro    = sig_mask('Pro-Social')
    sig_avoid  = sig_mask('Social Avoidance')

    def cell_color(c):
        sigs = sum([c in sig_agg, c in sig_pro, c in sig_avoid])
        if sigs == 0:
            return 'lightgray'
        if sigs > 1:
            return mixed_color
        if c in sig_agg:   return color_map['Aggressive']
        if c in sig_pro:   return color_map['Pro-Social']
        return color_map['Social Avoidance']
    
    red_cells = [c for c in cell_cols_z if cell_color(c) == color_map['Aggressive']]
    if red_cells:
        print(f"{animal} {session}: {len(red_cells)} red cells")

    
    colors = [cell_color(c) for c in cell_cols_z]
    ax.scatter(x, y, z, c=colors, alpha=0.7, s=20, edgecolors='none')

    handles = [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=color_map[bt], markersize=8, label=bt)
        for bt in CATEGORIES
    ] + [
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=mixed_color, markersize=8, label='Mixed'),
        plt.Line2D([0], [0], marker='o', color='w',
                   markerfacecolor='lightgray', markersize=8, label='n.s.'),
    ]
    ax.legend(handles=handles, frameon=False, fontsize=7,
              bbox_to_anchor=(1.1, 1), loc='upper left')

    ax.set_xlabel('Aggressive')
    ax.set_ylabel('Pro-Social')
    ax.set_zlabel('Social Avoidance')
    ax.set_title(f'{animal} {session}')
    plt.tight_layout()
    plt.show()


In [ ]:
CATEGORIES = ['Aggressive', 'Pro-Social', 'Social Avoidance']

In [ ]:
%matplotlib widget

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')

for (animal, session), grp in zdf.groupby(['animal', 'session']):
    grp_r = grp.set_index('behavior_type').reindex(CATEGORIES)
    cell_cols_z = [c for c in grp_r.columns if c not in ('animal', 'session')]

    x = grp_r.loc['Aggressive',       cell_cols_z].fillna(0).values.astype(float)
    y = grp_r.loc['Pro-Social',        cell_cols_z].fillna(0).values.astype(float)
    z = grp_r.loc['Social Avoidance',  cell_cols_z].fillna(0).values.astype(float)

    def sig_mask(bt):
        row = tuned[(tuned['animal'] == animal) & (tuned['session'] == session)
                    & (tuned['behavior_type'] == bt)]
        return set(row['tuned_cells'].iloc[0]) if not row.empty else set()

    sig_agg   = sig_mask('Aggressive')
    sig_pro   = sig_mask('Pro-Social')
    sig_avoid = sig_mask('Social Avoidance')

    def cell_color(c):
        sigs = sum([c in sig_agg, c in sig_pro, c in sig_avoid])
        if sigs == 0:      return 'lightgray'
        if sigs > 1:       return mixed_color
        if c in sig_agg:   return color_map['Aggressive']
        if c in sig_pro:   return color_map['Pro-Social']
        return color_map['Social Avoidance']

    colors = [cell_color(c) for c in cell_cols_z]
    ax.scatter(x, y, z, c=colors, alpha=0.7, s=5, edgecolors=None)

handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=color_map[bt], markersize=8, label=bt)
    for bt in CATEGORIES
] + [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=mixed_color, markersize=8, label='Mixed'),
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor='lightgray', markersize=8, label='n.s.'),
]
ax.legend(handles=handles, frameon=False, fontsize=7,
          bbox_to_anchor=(1.1, 1), loc='upper left')

ax.set_xlabel('Aggressive')
ax.set_ylabel('Pro-Social')
ax.set_zlabel('Social Avoidance')
ax.set_title('All animals')
ax.set_xlim(-4, 5)   # Aggressive
ax.set_ylim(-2, 5)   # Pro-Social
ax.set_zlim(-4, 5)   # Social Avoidance
xlim = ax.get_xlim()
ylim = ax.get_ylim()
zlim = ax.get_zlim()
ax.plot(xlim, [0, 0], [0, 0], color='gray', linewidth=0.8, linestyle='--')
ax.plot([0, 0], ylim, [0, 0], color='gray', linewidth=0.8, linestyle='--')
ax.plot([0, 0], [0, 0], zlim, color='gray', linewidth=0.8, linestyle='--')


ax.grid(False)

plt.tight_layout()
plt.show()

# 2D Scatter plots

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111)

for (animal, session), grp in zdf.groupby(['animal', 'session']):
    grp_r = grp.set_index('behavior_type').reindex(CATEGORIES)
    cell_cols_z = [c for c in grp_r.columns if c not in ('animal', 'session')]

    x = grp_r.loc['Pro-Social',        cell_cols_z].fillna(0).values.astype(float)
    y = grp_r.loc['Social Avoidance',  cell_cols_z].fillna(0).values.astype(float)

    def sig_mask(bt):
        row = tuned[(tuned['animal'] == animal) & (tuned['session'] == session)
                    & (tuned['behavior_type'] == bt)]
        return set(row['tuned_cells'].iloc[0]) if not row.empty else set()

    sig_agg   = sig_mask('Aggressive')
    sig_pro   = sig_mask('Pro-Social')
    sig_avoid = sig_mask('Social Avoidance')

    def cell_color(c):
        sigs = sum([c in sig_agg, c in sig_pro, c in sig_avoid])
        if sigs == 0:      return 'lightgray'
        if sigs > 1:       return mixed_color
        if c in sig_agg:   return color_map['Aggressive']
        if c in sig_pro:   return color_map['Pro-Social']
        return color_map['Social Avoidance']

    colors = [cell_color(c) for c in cell_cols_z]
    ax.scatter(x, y, c=colors, alpha=0.5, s=10, edgecolors='none')

handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=color_map[bt], markersize=8, label=bt)
    for bt in CATEGORIES
] + [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=mixed_color, markersize=8, label='Mixed'),
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor='lightgray', markersize=8, label='n.s.'),
]
ax.legend(handles=handles, frameon=False, fontsize=7,
          bbox_to_anchor=(1.1, 1), loc='upper left')

ax.set_xlabel('Pro-Social')
ax.set_ylabel('Social Avoidant')
ax.set_title('All animals')

# zero lines
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.axhline(2,  color='gray', linewidth=0.8, linestyle='--')
ax.axhline(-2, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(2,  color='gray', linewidth=0.8, linestyle='--')
ax.axvline(-2, color='gray', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()


In [ ]:
zdf

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111)
zagg =zdf[zdf['animal'].isin(agg_animals)]

for (animal, session), grp in zagg.groupby(['animal', 'session']):
    grp_r = grp.set_index('behavior_type').reindex(CATEGORIES)
    cell_cols_z = [c for c in grp_r.columns if c not in ('animal', 'session')]

    x = grp_r.loc['Pro-Social',        cell_cols_z].fillna(0).values.astype(float)
    y = grp_r.loc['Aggressive',  cell_cols_z].fillna(0).values.astype(float)

    def sig_mask(bt):
        row = tuned[(tuned['animal'] == animal) & (tuned['session'] == session)
                    & (tuned['behavior_type'] == bt)]
        return set(row['tuned_cells'].iloc[0]) if not row.empty else set()

    sig_agg   = sig_mask('Aggressive')
    sig_pro   = sig_mask('Pro-Social')
    sig_avoid = sig_mask('Social Avoidance')

    def cell_color(c):
        sigs = sum([c in sig_agg, c in sig_pro, c in sig_avoid])
        if sigs == 0:      return 'lightgray'
        if sigs > 1:       return mixed_color
        if c in sig_agg:   return color_map['Aggressive']
        if c in sig_pro:   return color_map['Pro-Social']
        return color_map['Social Avoidance']

    colors = [cell_color(c) for c in cell_cols_z]
    ax.scatter(x, y, c=colors, alpha=0.5, s=10, edgecolors='none')

handles = [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=color_map[bt], markersize=8, label=bt)
    for bt in CATEGORIES
] + [
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=mixed_color, markersize=8, label='Mixed'),
    plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor='lightgray', markersize=8, label='n.s.'),
]
ax.legend(handles=handles, frameon=False, fontsize=7,
          bbox_to_anchor=(1.1, 1), loc='upper left')

ax.set_xlabel('Pro-Social')
ax.set_ylabel('Aggressive')
ax.set_title('Aggressive animals')

# zero lines
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.axhline(2,  color='gray', linewidth=0.8, linestyle='--')
ax.axhline(-2, color='gray', linewidth=0.8, linestyle='--')
ax.axvline(2,  color='gray', linewidth=0.8, linestyle='--')
ax.axvline(-2, color='gray', linewidth=0.8, linestyle='--')

plt.tight_layout()
plt.show()


# Cluster maps 

In [ ]:
from io import BytesIO
from PIL import Image

def make_clustermap_img(animal, session,color_map):
    grp = tuned[(tuned['animal'] == animal) & (tuned['session'] == session)]

    # All significantly tuned cells for this animal/session
    all_cells = sorted(set(c for cells in grp['tuned_cells'] for c in cells))
    if len(all_cells) < 2:
        return None

    # Get z-scores for all 3 behaviors for every tuned cell
    rows = []
    for bt in CATEGORIES:
        z_row = zscores[
            (zscores['animal'] == animal) &
            (zscores['session'] == session) &
            (zscores['behavior_type'] == bt)
        ]
        if z_row.empty:
            rows.append(pd.Series(0.0, index=all_cells, name=bt))
        else:
            rows.append(z_row[all_cells].iloc[0].rename(bt))

    clust_df = pd.DataFrame(rows).fillna(0)

    col_colors = clust_df.idxmax(axis=0).map(color_map)
    g = sns.clustermap(clust_df, row_cluster=False, col_colors=col_colors,
                       cmap='RdBu_r', center=0, xticklabels=False,
                       figsize=(6, 3), cbar_kws={'label': 'z-score'})
    g.fig.suptitle(session, y=1.02)
    buf = BytesIO()
    g.fig.savefig(buf, format='png', bbox_inches='tight', dpi=120)
    plt.close(g.fig)
    buf.seek(0)
    return Image.open(buf)


animals = sorted(tuned['animal'].unique())
for animal in animals:
    img_r1 = make_clustermap_img(animal, 'R1',color_map_categories)
    img_r2 = make_clustermap_img(animal, 'R2',color_map_categories)
    imgs = [(img, lbl) for img, lbl in [(img_r1, 'R1'), (img_r2, 'R2')] if img is not None]
    if not imgs:
        continue
    fig, axes = plt.subplots(1, len(imgs), figsize=(7 * len(imgs), 5))
    if len(imgs) == 1:
        axes = [axes]
    for ax, (img, lbl) in zip(axes, imgs):
        ax.imshow(img)
        ax.axis('off')
    fig.suptitle(animal, fontsize=12)
    plt.tight_layout()
    plt.show()


## All animals pooled

In [ ]:
# ── All significantly tuned cells pooled, z-scores for all 3 behaviors ──
all_records = []
for (animal, session), grp in tuned.groupby(['animal', 'session']):
    all_cells = sorted(set(c for cells in grp['tuned_cells'] for c in cells))
    if not all_cells:
        continue
    for bt in CATEGORIES:
        z_row = zscores[
            (zscores['animal'] == animal) &
            (zscores['session'] == session) &
            (zscores['behavior_type'] == bt)
        ]
        for c in all_cells:
            val = float(z_row[c].iloc[0]) if not z_row.empty else 0.0
            all_records.append({
                'cell': f"{animal}_{session}_{c}",
                'behavior_type': bt,
                'zscore': val
            })

clust_df = (pd.DataFrame(all_records)
              .pivot_table(index='cell', columns='behavior_type', values='zscore', fill_value=0)
              .reindex(columns=CATEGORIES, fill_value=0)
              .T)

col_colors = clust_df.idxmax(axis=0).map(color_map_categories)

g = sns.clustermap(
    clust_df,
    metric = 'euclidean',
    row_cluster=False,
    col_colors=col_colors,
    cmap='RdBu_r',
    center=0,
    xticklabels=False,
    figsize=(10, 4),
    cbar_kws={'label': 'z-score'},
    cbar_pos=(0.1, 0.1, .03, .1))

plt.suptitle('Z-scores of significantly tuned cells — all animals')
plt.tight_layout()
plt.savefig('./figures/zscores_tunedcells_clustermap.svg', transparent=True)

# Ordered ridgeplot with behaviors

In [ ]:
intervals

In [ ]:
animal = 'C3F3'
session ='R2'

tuned_dict = get_exclusive_tuned(tuned,animal,session)
mixed_dict = get_mixed_tuned(tuned,animal,session)

cell_cols = [c for c in df.columns if c not in ('animal', 'session', 'behavior_type')]
intervals = build_intervals(animal, session)
act = load_aligned_act(animal,session,
                       '/Users/amonast/Desktop/miniscope/Traces',
                       '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv',
                       '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv',)

cell_cols = [c for c in act.columns if str(c).isdigit()]
act[cell_cols] = gaussian_filter1d(act[cell_cols].to_numpy(), sigma=4, axis=0)

plot_all_tuned(act,intervals,tuned,animal,session,intervals['behavior_type'].unique(),color_map=color_map,positive_only=True)

In [ ]:
plot_all_tuned_with_mixed(act,intervals,tuned,animal,session,color_map=color_map,positive_only=True)

# regression of activity for each cell social versus avoidant

In [ ]:
for ani in zdf.animal.unique():
    for session in ['R1','R2']:
        sub_z = zdf.loc[(zdf['animal']==ani)&(zdf['session']==session)]
        sub_z = (sub_z.set_index('behavior_type')
                    .drop(columns=['animal', 'session'])
                    .T)
        fig,ax = plt.subplots()
        sns.scatterplot(data=sub_z,x='Pro-Social',y='Social Avoidance',ax=ax)
        plt.show()

## all animals all cells

In [ ]:
zdf_long

In [ ]:
zdf_long = zdf.melt(id_vars=['animal', 'session', 'behavior_type'],
                    var_name='cell', value_name='zscore')
zdf_wide = zdf_long.pivot_table(index=['animal', 'session', 'cell'],
                                 columns='behavior_type',
                                 values='zscore').reset_index()

fig, ax = plt.subplots()
sns.regplot(data=zdf_wide, x='Pro-Social', y='Social Avoidance', ax=ax, line_kws=dict(color="r"))
plt.show()

In [ ]:
from scipy import stats
res = stats.linregress(zdf_wide['Pro-Social'].dropna(),
                       zdf_wide['Social Avoidance'].dropna())
ax.set_title(f'r = {res.rvalue:.3f}, p = {res.pvalue:.3e}')


# Heatmap of the intruder entry

In [ ]:
PRE_SEC    = 30
WINDOW_SEC = 120
N_POINTS   = 20000  # common time grid resolution

t_common = np.linspace(-PRE_SEC, WINDOW_SEC, N_POINTS)

all_traces = []  # list of (N_cells, N_points) arrays
all_labels = []  # (animal, session, cell) per row

sessions_list = sorted(tuned[['animal','session']].drop_duplicates().itertuples(index=False))

for ani, session in sessions_list:
    act_ani = load_aligned_act(ani, session,
                               '/Users/amonast/Desktop/miniscope/Traces',
                               '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv',
                               '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv')
    intervals_ani = build_intervals(ani, session)

    # act index is timedelta — convert to seconds for the function
    act_sec = act_ani.loc[(ani, session)].copy()
    act_sec.index = act_sec.index.total_seconds()

    t_rel, window = get_first_prosocial_window(intervals_ani, act_sec,
                                               window_sec=WINDOW_SEC, pre_sec=PRE_SEC)
    if t_rel is None:
        continue

    cell_cols = [c for c in act_sec.columns if str(c).isdigit()]
    interp = np.vstack([
        np.interp(t_common, t_rel, window[:, i])
        for i in range(window.shape[1])
    ])  # (N_cells, N_points)

    all_traces.append(interp)
    all_labels.extend([(ani, session, c) for c in cell_cols])

pop_matrix = np.vstack(all_traces)
labels_df  = pd.DataFrame(all_labels, columns=['animal', 'session', 'cell'])


In [ ]:
# Sort by integrated activity in first 30s of interaction (t=0 to t=30)
SORT_WINDOW = 30.0
sort_mask = (t_common >= 0) & (t_common <= SORT_WINDOW)
integrated = pop_matrix[:, sort_mask].sum(axis=1)
sort_order = np.argsort(integrated)[::-1]
sorted_mat = pop_matrix[sort_order]

n_ticks = 7
tick_idx = np.linspace(0, N_POINTS - 1, n_ticks, dtype=int)
tick_labels = [f'{t_common[i]:.0f}' for i in tick_idx]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    sorted_mat,
    ax=ax,
    cmap='RdBu_r',
    center=0,
    xticklabels=False,
    vmax=6,
    yticklabels=False,
    cbar_kws={'label': 'z-score', 'shrink': 0.5}
)
ax.set_xticks(tick_idx)
ax.set_xticklabels(tick_labels, rotation=0)
onset_x = np.interp(0, t_common, np.arange(N_POINTS))
ax.axvline(onset_x, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Time relative to first Pro-Social onset (s)')
ax.set_ylabel('Cell (sorted by integrated activity 0–30s)')
ax.set_title('Pseudo-population heatmap — first Pro-Social encounter')
plt.tight_layout()
plt.show()


# Per-animal sorted population heatmap with behavior raster

In [ ]:
from matplotlib.gridspec import GridSpec

PRE_SEC_IND     = 30
WINDOW_SEC_IND  = 120
N_POINTS_IND    = 2000
SORT_WINDOW_IND = 30.0
t_common_ind    = np.linspace(-PRE_SEC_IND, WINDOW_SEC_IND, N_POINTS_IND)

def time_to_pix(t):
    return np.interp(t, t_common_ind, np.arange(N_POINTS_IND))

sessions_list = sorted(tuned[['animal', 'session']].drop_duplicates().itertuples(index=False))

for ani, session in sessions_list:
    act_ani = load_aligned_act(ani, session,
                               '/Users/amonast/Desktop/miniscope/Traces',
                               '/Users/amonast/Desktop/miniscope/data/tuning/intervals_allmice.csv',
                               '/Users/amonast/Desktop/miniscope/dbnst_data_info.csv')
    intervals_ani = build_intervals(ani, session)

    act_sec = act_ani.loc[(ani, session)].copy()
    act_sec.index = act_sec.index.total_seconds()
    cell_cols_ani = [c for c in act_sec.columns if str(c).isdigit() or isinstance(c, (int, np.integer))]
    act_sec[cell_cols_ani] = gaussian_filter1d(act_sec[cell_cols_ani].to_numpy(), sigma=4, axis=0)

    t_rel, window = get_first_prosocial_window(intervals_ani, act_sec,
                                               window_sec=WINDOW_SEC_IND, pre_sec=PRE_SEC_IND)
    if t_rel is None:
        print(f'{ani} {session}: no Pro-Social onset, skipping')
        continue

    onset = intervals_ani[intervals_ani['behavior_type'] == 'Pro-Social'].sort_values('start').iloc[0]['start']

    interp = np.vstack([
        np.interp(t_common_ind, t_rel, window[:, i])
        for i in range(window.shape[1])
    ])

    sort_mask  = (t_common_ind >= 0) & (t_common_ind <= SORT_WINDOW_IND)
    sort_order = np.argsort(interp[:, sort_mask].sum(axis=1))[::-1]
    sorted_mat = interp[sort_order]

    ivs_rel = intervals_ani.copy()
    ivs_rel['start_rel'] = ivs_rel['start'] - onset
    ivs_rel['end_rel']   = ivs_rel['end']   - onset
    ivs_window = ivs_rel[
        (ivs_rel['end_rel']   > -PRE_SEC_IND) &
        (ivs_rel['start_rel'] < WINDOW_SEC_IND) &
        (ivs_rel['behavior_type'].isin(color_map))
    ]

    # ── Layout: behavior | heatmap | colorbar ────────────────────────────────
    fig = plt.figure(figsize=(11, 6))
    gs  = GridSpec(2, 2, figure=fig,
                   height_ratios=[1, 8], width_ratios=[20, 1],
                   hspace=0.05, wspace=0.3)
    ax_heat = fig.add_subplot(gs[1, 0])
    ax_beh  = fig.add_subplot(gs[0, 0], sharex=ax_heat)   # shared x
    ax_cbar = fig.add_subplot(gs[1, 1])

    # ── Heatmap ───────────────────────────────────────────────────────────────
    n_ticks     = 7
    tick_idx    = np.linspace(0, N_POINTS_IND - 1, n_ticks, dtype=int)
    tick_labels = [f'{t_common_ind[i]:.0f}' for i in tick_idx]

    sns.heatmap(sorted_mat, ax=ax_heat, cmap='RdBu_r', center=0,
                xticklabels=False, vmax=6, yticklabels=False, cbar=False)
    fig.colorbar(ax_heat.collections[0], cax=ax_cbar, label='z-score')

    ax_heat.set_xticks(tick_idx)
    ax_heat.set_xticklabels(tick_labels, rotation=0)
    onset_pix = time_to_pix(0)
    ax_heat.axvline(onset_pix, color='black', linewidth=1, linestyle='--')
    ax_heat.set_xlabel('Time relative to first Pro-Social onset (s)')
    ax_heat.set_ylabel('Cell (sorted by 0–30s activity)')

    # ── Behavior raster (x shared with heatmap) ───────────────────────────────
    seen = set()
    for _, row in ivs_window.iterrows():
        bt = row['behavior_type']
        x0 = time_to_pix(row['start_rel'])
        x1 = time_to_pix(row['end_rel'])
        ax_beh.barh(y=0, width=x1 - x0, left=x0, height=0.8,
                    color=color_map[bt], label=bt if bt not in seen else '', alpha=0.8)
        seen.add(bt)
    ax_beh.axvline(onset_pix, color='black', linewidth=1, linestyle='--')
    ax_beh.set_ylim([-0.5, 0.5])
    ax_beh.set_yticks([])
    plt.setp(ax_beh.get_xticklabels(), visible=False)
    ax_beh.set_ylabel('Behavior', fontsize=8, rotation=0, labelpad=45, va='center')
    ax_beh.spines[['top', 'right', 'left', 'bottom']].set_visible(False)
    ax_beh.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)

    fig.suptitle(f'{ani} {session} — n={sorted_mat.shape[0]} cells')
    plt.tight_layout()
    plt.show()


Tuned ETAS

In [ ]:
from matplotlib.gridspec import GridSpec
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42   # restore your cell-0 setting, which 'default' resets

def plot_tuned_eta(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                   s_post=3, direction=None, smooth_sigma=4, fps=30,
                   merge_gap=0.4, color_map=None, vmax=2, save_path=None):
    """
    Event-triggered average of cells tuned to `category`, aligned to bout onset.

    A bout is included if:
      - behavior_type == category
      - duration >= bout_len (s)
      - no scored behavior of ANY type in the s_pre seconds before onset
      - the window [-s_pre, s_post] lies within the recording
    Contiguous intervals of the same category (gap <= merge_gap) are merged into
    one bout first (e.g. sniff -> follow counts as a single Pro-Social bout).

    animals   : list of animal IDs, or None for all animals with tuned cells
    direction : None (all tuned cells), 'excited' (z>0) or 'inhibited' (z<0)

    Each cell's trace is averaged over its own animal's qualifying bouts. The plot
    shows those per-cell averages (heatmap) and the mean ± SEM across cells.
    Returns dict(t, traces, labels, n_bouts).
    """
    if color_map is None:
        color_map = color_map_categories
    color = color_map.get(category, '#333333')

    rows = tuned[(tuned['behavior_type'] == category) & (tuned['session'] == session)]
    if animals is not None:
        rows = rows[rows['animal'].isin(animals)]

    t_rel = np.arange(-s_pre, s_post, 1 / fps)
    traces, labels, n_bouts = [], [], {}

    for _, r in rows.iterrows():
        ani   = r['animal']
        cells = list(r['tuned_cells'])

        if direction is not None:
            z_row = zscores[(zscores['animal'] == ani) &
                            (zscores['session'] == session) &
                            (zscores['behavior_type'] == category)]
            if z_row.empty:
                continue
            z = z_row[cells].iloc[0].astype(float)
            cells = [c for c in cells if (z[c] > 0 if direction == 'excited' else z[c] < 0)]
        if not cells:
            continue

        # ── bouts ────────────────────────────────────────────────────────────
        ivs = build_intervals(ani, session).sort_values('start').reset_index(drop=True)
        bouts = []
        for s, e in ivs.loc[ivs['behavior_type'] == category, ['start', 'end']].to_numpy():
            if bouts and s - bouts[-1][1] <= merge_gap:
                bouts[-1][1] = max(bouts[-1][1], e)
            else:
                bouts.append([s, e])

        # ── activity ─────────────────────────────────────────────────────────
        act_ani = load_aligned_act(ani, session, traces_dir, None, data_info_csv)
        sub = act_ani.loc[(ani, session)]
        t   = sub.index.total_seconds().to_numpy()
        X   = sub[cells].to_numpy(dtype=float)
        if smooth_sigma:
            X = gaussian_filter1d(X, sigma=smooth_sigma, axis=0)

        onsets = []
        for s, e in bouts:
            if e - s < bout_len:
                continue
            if ((ivs['start'] < s) & (ivs['end'] > s - s_pre)).any():
                continue
            if s - s_pre < t[0] or s + s_post > t[-1]:
                continue
            onsets.append(s)

        if not onsets:
            print(f'{ani} {session}: no qualifying {category} bouts')
            continue
        n_bouts[ani] = len(onsets)

        # linear interpolation onto common grid, vectorized over cells
        frame_idx = np.arange(len(t))
        stack = []
        for o in onsets:
            fi = np.interp(o + t_rel, t, frame_idx)
            lo = np.clip(np.floor(fi).astype(int), 0, len(t) - 2)
            w  = (fi - lo)[:, None]
            stack.append(X[lo] * (1 - w) + X[lo + 1] * w)   # (n_time, n_cells)
        eta = np.mean(stack, axis=0).T                      # (n_cells, n_time)

        traces.append(eta)
        labels.extend([(ani, session, c) for c in cells])

    if not traces:
        print(f'No tuned cells with qualifying {category} bouts in {session}')
        return None

    traces = np.vstack(traces)
    labels = pd.DataFrame(labels, columns=['animal', 'session', 'cell'])

    # sort by mean response during the guaranteed in-bout window [0, bout_len]
    resp_mask = (t_rel >= 0) & (t_rel <= min(bout_len, s_post))
    order = np.argsort(traces[:, resp_mask].mean(axis=1))[::-1]

    mean = traces.mean(axis=0)
    sem  = traces.std(axis=0, ddof=1) / np.sqrt(len(traces)) if len(traces) > 1 else np.zeros_like(mean)

       # ── plot ─────────────────────────────────────────────────────────────────
    with plt.rc_context({'font.size': 5, 'axes.labelsize': 4, 'xtick.labelsize': 4,
                         'ytick.labelsize': 4, 'axes.linewidth': .5,
                         'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
                         'xtick.major.size': 1, 'ytick.major.size': 1}):
        fig = plt.figure(figsize=(1, 2.3, dpi=300)
        gs  = GridSpec(2, 2, figure=fig, height_ratios=[1.7,1], width_ratios=[20, 1],
                       hspace=0.1, wspace=0.05, left=0.16, right=0.86, top=0.84, bottom=0.14)
        ax_heat = fig.add_subplot(gs[0, 0])
        ax_mean = fig.add_subplot(gs[1, 0], sharex=ax_heat)
        ax_cbar = fig.add_subplot(gs[0, 1])

        im = ax_heat.imshow(traces[order], aspect='auto', cmap='RdBu_r',
                            vmin=-vmax, vmax=vmax, interpolation='none',
                            extent=[t_rel[0], t_rel[-1], len(traces), 0])
        cb = fig.colorbar(im, cax=ax_cbar)
        cb.set_label('z-score', labelpad=2)
        cb.outline.set_linewidth(0.5)
        cb.locator = matplotlib.ticker.MaxNLocator(integer=True)
        cb.formatter = matplotlib.ticker.FormatStrFormatter('%d')
        cb.update_ticks()
        ax_heat.axvline(0, color='k', linestyle='--', linewidth=0.5)
        ax_heat.set_ylabel('Cell', labelpad=2)
        ax_heat.set_yticks([0, len(traces)])
        ax_heat.set_yticklabels(['1', str(len(traces))])
        plt.setp(ax_heat.get_xticklabels(), visible=False)

        ax_mean.plot(t_rel, mean, color=color, linewidth=0.8)
        ax_mean.fill_between(t_rel, mean - sem, mean + sem, color=color, alpha=0.3, linewidth=0)
        ax_mean.axvline(0, color='k', linestyle='--', linewidth=0.5)
        ax_mean.axhline(0, color='gray', linewidth=0.3)
        #ax_mean.set_xlabel(f'Time from {category} onset (s)', labelpad=2)
        #ax_mean.set_ylabel('z-score', labelpad=2)
        ax_mean.spines[['top', 'right']].set_visible(False)

        dir_lbl = f' ({direction})' if direction else ''
        # fig.suptitle(f'{category}{dir_lbl} — {session}\n'
        #              f'{len(traces)} cells, {sum(n_bouts.values())} bouts, {len(n_bouts)} animals',
        #              fontsize=6, y=0.97)

        if save_path:
            fig.savefig(save_path, transparent=True)
        #plt.tight_layout()

    return #dict(t=t_rel, traces=traces, labels=labels, n_bouts=n_bouts)

In [ ]:
plot_tuned_eta('Pro-Social', bout_len=.5, s_pre=.5, session='R2',direction='excited',
               save_path='figures/prosocial_eta_heatmap.pdf')


In [ ]:
plot_tuned_eta('Aggressive', bout_len=.5, s_pre=.5, session='R2',direction='excited',
                save_path='figures/aggressive_eta_heatmap.pdf')


In [ ]:
plot_tuned_eta('Social Avoidance', bout_len=.5, s_pre=.5, session='R2',direction='excited',
               save_path='figures/avoidance_eta_heatmap.pdf')

# normalize time

In [ ]:
def _collect_eta_warped(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                        max_bout_len=None, s_post=1, n_bins=50, direction=None,
                        smooth_sigma=4, fps=30, merge_gap=0.4,
                        plot_pre=None, plot_post=None):
    """
    Per-cell time-normalized ETAs.

    Inclusion: s_pre = behavior-free window before onset (criterion only).
    Plotting:  plot_pre / plot_post = seconds shown before onset / after offset
               (default s_pre / s_post). Bouts must also have the full plotted
               window inside the recording.
    Returns (x, traces, labels, n_bouts, durations, med) or None.
    """
    plot_pre  = s_pre  if plot_pre  is None else plot_pre
    plot_post = s_post if plot_post is None else plot_post

    rows = tuned[(tuned['behavior_type'] == category) & (tuned['session'] == session)]
    if animals is not None:
        rows = rows[rows['animal'].isin(animals)]

    pre_t  = np.arange(-plot_pre, 0, 1 / fps)
    post_t = np.arange(1, round(plot_post * fps) + 1) / fps
    u      = (np.arange(n_bins) + 0.5) / n_bins

    traces, labels, n_bouts, durations = [], [], {}, []

    for _, r in rows.iterrows():
        ani   = r['animal']
        cells = list(r['tuned_cells'])

        if direction is not None:
            z_row = zscores[(zscores['animal'] == ani) &
                            (zscores['session'] == session) &
                            (zscores['behavior_type'] == category)]
            if z_row.empty:
                continue
            z = z_row[cells].iloc[0].astype(float)
            cells = [c for c in cells if (z[c] > 0 if direction == 'excited' else z[c] < 0)]
        if not cells:
            continue

        ivs = build_intervals(ani, session).sort_values('start').reset_index(drop=True)
        bouts = []
        for s, e in ivs.loc[ivs['behavior_type'] == category, ['start', 'end']].to_numpy():
            if bouts and s - bouts[-1][1] <= merge_gap:
                bouts[-1][1] = max(bouts[-1][1], e)
            else:
                bouts.append([s, e])

        act_ani = load_aligned_act(ani, session, traces_dir, None, data_info_csv)
        sub = act_ani.loc[(ani, session)]
        t   = sub.index.total_seconds().to_numpy()
        X   = sub[cells].to_numpy(dtype=float)
        if smooth_sigma:
            X = gaussian_filter1d(X, sigma=smooth_sigma, axis=0)

        stack = []
        for s, e in bouts:
            dur = e - s
            if dur < bout_len or (max_bout_len is not None and dur > max_bout_len):
                continue
            if ((ivs['start'] < s) & (ivs['end'] > s - s_pre)).any():       # inclusion
                continue
            if s - plot_pre < t[0] or e + plot_post > t[-1]:                 # plot window
                continue
            query = np.concatenate([s + pre_t, s + u * dur, e + post_t])
            stack.append(_interp_rows(X, t, query))
            durations.append(dur)

        if not stack:
            print(f'{ani} {session}: no qualifying {category} bouts')
            continue
        n_bouts[ani] = len(stack)
        traces.append(np.mean(stack, axis=0).T)
        labels.extend([(ani, session, c) for c in cells])

    if not traces:
        print(f'No tuned cells with qualifying {category} bouts in {session}')
        return None

    med = np.median(durations)
    x   = np.concatenate([pre_t / med, u, 1 + post_t / med])
    labels = pd.DataFrame(labels, columns=['animal', 'session', 'cell'])
    return x, np.vstack(traces), labels, n_bouts, np.array(durations), med


def plot_tuned_eta_warped(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                          max_bout_len=None, s_post=1, n_bins=200, direction=None,
                          smooth_sigma=2, fps=30, merge_gap=0.4, color_map=None,
                          vmax=2, plot_pre=None, plot_post=None, save_path=None):
    """
    Time-normalized ETA heatmap + mean ± SEM.
    s_pre             : behavior-free window before onset (inclusion only)
    plot_pre/plot_post: seconds shown before onset / after offset
                        (default s_pre / s_post)
    If plot_pre > s_pre, a dotted line marks where the clean window begins.
    """
    plot_pre  = s_pre  if plot_pre  is None else plot_pre
    plot_post = s_post if plot_post is None else plot_post

    out = _collect_eta_warped(category, bout_len, s_pre, animals, session, max_bout_len,
                              s_post, n_bins, direction, smooth_sigma, fps, merge_gap,
                              plot_pre=plot_pre, plot_post=plot_post)
    if out is None:
        return None
    x, traces, labels, n_bouts, durations, med = out

    if color_map is None:
        color_map = color_map_categories
    color = color_map.get(category, '#333333')

    in_bout = (x >= 0) & (x <= 1)
    order   = np.argsort(traces[:, in_bout].mean(axis=1))[::-1]

    mean = traces.mean(axis=0)
    sem  = traces.std(axis=0, ddof=1) / np.sqrt(len(traces)) if len(traces) > 1 else np.zeros_like(mean)

    xticks, xlabels = [0, 1], ['on', 'off']
    if plot_pre > 0:
        xticks.insert(0, -plot_pre / med);  xlabels.insert(0, f'-{plot_pre:g}s')
    if plot_post > 0:
        xticks.append(1 + plot_post / med); xlabels.append(f'+{plot_post:g}s')

    with plt.rc_context({'font.size': 6, 'axes.labelsize': 6, 'xtick.labelsize': 5,
                         'ytick.labelsize': 5, 'axes.linewidth': 0.5,
                         'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
                         'xtick.major.size': 2, 'ytick.major.size': 2}):
        fig = plt.figure(figsize=(3, 3), dpi=300)
        gs  = GridSpec(2, 2, figure=fig, height_ratios=[3, 1.5], width_ratios=[20, 1],
                       hspace=0.1, wspace=0.05, left=0.16, right=0.86, top=0.84, bottom=0.14)
        ax_heat = fig.add_subplot(gs[0, 0])
        ax_mean = fig.add_subplot(gs[1, 0], sharex=ax_heat)
        ax_cbar = fig.add_subplot(gs[0, 1])

        im = ax_heat.pcolormesh(x, np.arange(len(traces)), traces[order],
                                cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                                shading='nearest', rasterized=True)
        ax_heat.invert_yaxis()
        cb = fig.colorbar(im, cax=ax_cbar)
        cb.set_label('z-score', labelpad=2)
        cb.outline.set_linewidth(0.5)
        ax_heat.set_ylabel('Cell', labelpad=2)
        plt.setp(ax_heat.get_xticklabels(), visible=False)

        ax_mean.axvspan(0, 1, color=color, alpha=0.1, linewidth=0)
        ax_mean.plot(x, mean, color=color, linewidth=0.8)
        ax_mean.fill_between(x, mean - sem, mean + sem, color=color, alpha=0.3, linewidth=0)
        ax_mean.axhline(0, color='gray', linewidth=0.3)

        for ax in (ax_heat, ax_mean):
            for xv in (0, 1):
                ax.axvline(xv, color='k', linestyle='--', linewidth=0.5)
            if plot_pre > s_pre:                       # start of behavior-free window
                ax.axvline(-s_pre / med, color='gray', linestyle=':', linewidth=0.5)

        ax_mean.set_xticks(xticks)
        ax_mean.set_xticklabels(xlabels)
        ax_mean.set_xlim(x[0], x[-1])
        ax_mean.set_xlabel(f'{category} bout (normalized)', labelpad=2)
        ax_mean.set_ylabel('z-score', labelpad=2)
        
        ax_mean.spines[['top', 'right']].set_visible(False)

        dir_lbl = f' ({direction})' if direction else ''
        fig.suptitle(f'{category}{dir_lbl} — {session}\n'
                     f'{len(traces)} cells, {sum(n_bouts.values())} bouts, '
                     f'{len(n_bouts)} animals, median {med:.1f}s',
                     fontsize=6, y=0.97)

        if save_path:
            fig.savefig(save_path, transparent=True)
        plt.show()

    return dict(x=x, traces=traces, labels=labels, n_bouts=n_bouts, durations=durations)

In [ ]:

res = plot_tuned_eta_warped('Social Avoidance', bout_len=.3, s_pre=2, session='R2',
                            direction='excited', save_path='figures/avoidant_eta_warped.png')

In [ ]:
plot_tuned_eta_warped('Aggressive', bout_len=.3, s_pre=.5, plot_pre=1, plot_post=0,
                      direction='excited',save_path='figures/aggressive_eta_warped.png')

In [ ]:
res = plot_tuned_eta_warped('Pro-Social', bout_len=1, s_pre=2, session='R2',
                            direction='excited', save_path='figures/aggressive_eta_warped.png')

In [ ]:
def _collect_eta(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                 s_post=3, direction=None, smooth_sigma=4, fps=30, merge_gap=0.4):
    """
    Per-cell onset-aligned ETAs for cells tuned to `category`.
    Same bout criteria as plot_tuned_eta. Returns (t_rel, traces, labels, n_bouts)
    or None if nothing qualifies. traces: (n_cells, n_time).
    """
    rows = tuned[(tuned['behavior_type'] == category) & (tuned['session'] == session)]
    if animals is not None:
        rows = rows[rows['animal'].isin(animals)]

    t_rel = np.arange(-s_pre, s_post, 1 / fps)
    traces, labels, n_bouts = [], [], {}

    for _, r in rows.iterrows():
        ani   = r['animal']
        cells = list(r['tuned_cells'])

        if direction is not None:
            z_row = zscores[(zscores['animal'] == ani) &
                            (zscores['session'] == session) &
                            (zscores['behavior_type'] == category)]
            if z_row.empty:
                continue
            z = z_row[cells].iloc[0].astype(float)
            cells = [c for c in cells if (z[c] > 0 if direction == 'excited' else z[c] < 0)]
        if not cells:
            continue

        ivs = build_intervals(ani, session).sort_values('start').reset_index(drop=True)
        bouts = []
        for s, e in ivs.loc[ivs['behavior_type'] == category, ['start', 'end']].to_numpy():
            if bouts and s - bouts[-1][1] <= merge_gap:
                bouts[-1][1] = max(bouts[-1][1], e)
            else:
                bouts.append([s, e])

        act_ani = load_aligned_act(ani, session, traces_dir, None, data_info_csv)
        sub = act_ani.loc[(ani, session)]
        t   = sub.index.total_seconds().to_numpy()
        X   = sub[cells].to_numpy(dtype=float)
        if smooth_sigma:
            X = gaussian_filter1d(X, sigma=smooth_sigma, axis=0)

        onsets = [s for s, e in bouts
                  if e - s >= bout_len
                  and not ((ivs['start'] < s) & (ivs['end'] > s - s_pre)).any()
                  and s - s_pre >= t[0] and s + s_post <= t[-1]]
        if not onsets:
            print(f'{ani} {session}: no qualifying {category} bouts')
            continue
        n_bouts[ani] = len(onsets)

        stack = [_interp_rows(X, t, o + t_rel) for o in onsets]   # each (n_time, n_cells)
        traces.append(np.mean(stack, axis=0).T)
        labels.extend([(ani, session, c) for c in cells])

    if not traces:
        print(f'No tuned cells with qualifying {category} bouts in {session}')
        return None
    labels = pd.DataFrame(labels, columns=['animal', 'session', 'cell'])
    return t_rel, np.vstack(traces), labels, n_bouts


def plot_tuned_eta_ridge(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                         s_post=3, direction=None, smooth_sigma=4, fps=30,
                         merge_gap=0.4, color_map=None, max_cells=None,
                         trace_spacing=.3, scale_bar=1, save_path=None):
    """
    Same as plot_tuned_eta but with a ridge plot of per-cell ETAs instead of a heatmap.
    Cells are sorted by mean response in [0, bout_len], strongest at the top.
    max_cells : if set, plot only the top-N responders in the ridge (mean ± SEM
                panel still uses all cells).
    scale_bar : height in z-score of the vertical scale bar.
    """
    out = _collect_eta(category, bout_len, s_pre, animals, session, s_post,
                       direction, smooth_sigma, fps, merge_gap)
    if out is None:
        return None
    t_rel, traces, labels, n_bouts = out

    if color_map is None:
        color_map = color_map_categories
    color = color_map.get(category, '#333333')

    resp_mask = (t_rel >= 0) & (t_rel <= min(bout_len, s_post))
    order = np.argsort(traces[:, resp_mask].mean(axis=1))[::-1]     # strongest first
    shown = order[:max_cells] if max_cells else order
    ridge = traces[shown[::-1]]          # ridge_plot draws row 0 at the bottom

    mean = traces.mean(axis=0)
    sem  = traces.std(axis=0, ddof=1) / np.sqrt(len(traces)) if len(traces) > 1 else np.zeros_like(mean)

    with plt.rc_context({'font.size': 6, 'axes.labelsize': 6, 'xtick.labelsize': 5,
                         'ytick.labelsize': 5, 'axes.linewidth': 0.5,
                         'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
                         'xtick.major.size': 2, 'ytick.major.size': 2}):
        fig, (ax_ridge, ax_mean) = plt.subplots(
            2, 1, figsize=(3, 3), dpi=300, sharex=True,
            gridspec_kw=dict(height_ratios=[3, 1.5], hspace=0.1,
                             left=0.16, right=0.95, top=0.84, bottom=0.14))

        ridge_plot(ridge, t_rel, color=color, ax=ax_ridge, alpha=0.9,
                   line_width=0.5, trace_spacing=trace_spacing)
        ax_ridge.axvline(0, color='k', linestyle='--', linewidth=0.5)
        ax_ridge.set_yticks([])
        ax_ridge.spines[['left', 'bottom']].set_visible(False)
        ax_ridge.tick_params(axis='x', length=0)
        plt.setp(ax_ridge.get_xticklabels(), visible=False)
        ax_ridge.set_ylabel('Cell', labelpad=2)

        # vertical z-score scale bar, just past the right edge of the traces
        x_sb = t_rel[-1] + 0.02 * (t_rel[-1] - t_rel[0])
        ax_ridge.plot([x_sb, x_sb], [1, 1 + scale_bar], color='k', linewidth=0.8,
                      clip_on=False)
        ax_ridge.text(x_sb, 1 + scale_bar / 2, f' {scale_bar:g} z', va='center',
                      ha='left', fontsize=5)

        ax_mean.plot(t_rel, mean, color=color, linewidth=0.8)
        ax_mean.fill_between(t_rel, mean - sem, mean + sem, color=color, alpha=0.3, linewidth=0)
        ax_mean.axvline(0, color='k', linestyle='--', linewidth=0.5)
        ax_mean.axhline(0, color='gray', linewidth=0.3)
        ax_mean.set_xlim(t_rel[0], t_rel[-1])
        ax_mean.set_xlabel(f'Time from {category} onset (s)', labelpad=2)
        ax_mean.set_ylabel('z-score', labelpad=2)
        ax_mean.spines[['top', 'right']].set_visible(False)

        dir_lbl  = f' ({direction})' if direction else ''
        cell_lbl = (f'top {len(shown)} of {len(traces)} cells' if max_cells
                    else f'{len(traces)} cells')
        fig.suptitle(f'{category}{dir_lbl} — {session}\n'
                     f'{cell_lbl}, {sum(n_bouts.values())} bouts, {len(n_bouts)} animals',
                     fontsize=6, y=0.97)

        if save_path:
            fig.savefig(save_path, transparent=True)
        plt.show()

    return dict(t=t_rel, traces=traces, labels=labels, n_bouts=n_bouts, order=order)

In [ ]:
res = plot_tuned_eta_ridge('Aggressive', bout_len=.5, s_pre=.5, session='R2',
                           direction='excited', max_cells=30,
                           save_path='figures/aggressive_eta_ridge.svg')

In [ ]:
def _collect_eta_warped(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                        max_bout_len=None, s_post=1, n_bins=100, direction=None,
                        smooth_sigma=4, fps=30, merge_gap=0.4):
    """
    Per-cell time-normalized ETAs (same criteria as plot_tuned_eta_warped).
    Returns (x, traces, labels, n_bouts, durations, med) or None.
    x is the composite axis: pre/post in units of median bout duration, bout in [0, 1].
    """
    rows = tuned[(tuned['behavior_type'] == category) & (tuned['session'] == session)]
    if animals is not None:
        rows = rows[rows['animal'].isin(animals)]

    pre_t  = np.arange(-s_pre, 0, 1 / fps)
    post_t = np.arange(1, round(s_post * fps) + 1) / fps
    u      = (np.arange(n_bins) + 0.5) / n_bins

    traces, labels, n_bouts, durations = [], [], {}, []

    for _, r in rows.iterrows():
        ani   = r['animal']
        cells = list(r['tuned_cells'])

        if direction is not None:
            z_row = zscores[(zscores['animal'] == ani) &
                            (zscores['session'] == session) &
                            (zscores['behavior_type'] == category)]
            if z_row.empty:
                continue
            z = z_row[cells].iloc[0].astype(float)
            cells = [c for c in cells if (z[c] > 0 if direction == 'excited' else z[c] < 0)]
        if not cells:
            continue

        ivs = build_intervals(ani, session).sort_values('start').reset_index(drop=True)
        bouts = []
        for s, e in ivs.loc[ivs['behavior_type'] == category, ['start', 'end']].to_numpy():
            if bouts and s - bouts[-1][1] <= merge_gap:
                bouts[-1][1] = max(bouts[-1][1], e)
            else:
                bouts.append([s, e])

        act_ani = load_aligned_act(ani, session, traces_dir, None, data_info_csv)
        sub = act_ani.loc[(ani, session)]
        t   = sub.index.total_seconds().to_numpy()
        X   = sub[cells].to_numpy(dtype=float)
        if smooth_sigma:
            X = gaussian_filter1d(X, sigma=smooth_sigma, axis=0)

        stack = []
        for s, e in bouts:
            dur = e - s
            if dur < bout_len or (max_bout_len is not None and dur > max_bout_len):
                continue
            if ((ivs['start'] < s) & (ivs['end'] > s - s_pre)).any():
                continue
            if s - s_pre < t[0] or e + s_post > t[-1]:
                continue
            query = np.concatenate([s + pre_t, s + u * dur, e + post_t])
            stack.append(_interp_rows(X, t, query))
            durations.append(dur)

        if not stack:
            print(f'{ani} {session}: no qualifying {category} bouts')
            continue
        n_bouts[ani] = len(stack)
        traces.append(np.mean(stack, axis=0).T)
        labels.extend([(ani, session, c) for c in cells])

    if not traces:
        print(f'No tuned cells with qualifying {category} bouts in {session}')
        return None

    med = np.median(durations)
    x   = np.concatenate([pre_t / med, u, 1 + post_t / med])
    labels = pd.DataFrame(labels, columns=['animal', 'session', 'cell'])
    return x, np.vstack(traces), labels, n_bouts, np.array(durations), med


def plot_tuned_eta_warped_ridge(category, bout_len=.5, s_pre=1, animals=None, session='R2',
                                max_bout_len=None, s_post=1, n_bins=200, direction=None,
                                smooth_sigma=1, fps=30, merge_gap=0.4, color_map=None,
                                max_cells=None, trace_spacing=.2, scale_bar=2,
                                save_path=None):
    """
    Ridge-plot version of plot_tuned_eta_warped.
    Cells sorted by mean in-bout response (x in [0, 1]), strongest at the top.
    max_cells : top-N responders in the ridge; mean ± SEM panel uses all cells.
    """
    out = _collect_eta_warped(category, bout_len, s_pre, animals, session, max_bout_len,
                              s_post, n_bins, direction, smooth_sigma, fps, merge_gap)
    if out is None:
        return None
    x, traces, labels, n_bouts, durations, med = out

    if color_map is None:
        color_map = color_map_categories
    color = color_map.get(category, '#333333')

    in_bout = (x >= 0) & (x <= 1)
    order = np.argsort(traces[:, in_bout].mean(axis=1))[::-1]
    shown = order[:max_cells] if max_cells else order
    ridge = traces[shown[::-1]]            # ridge_plot draws row 0 at the bottom

    mean = traces.mean(axis=0)
    sem  = traces.std(axis=0, ddof=1) / np.sqrt(len(traces)) if len(traces) > 1 else np.zeros_like(mean)

    xticks  = [-s_pre / med, 0, 1, 1 + s_post / med]
    xlabels = [f'-{s_pre:g}s', 'on', 'off', f'+{s_post:g}s']

    with plt.rc_context({'font.size': 6, 'axes.labelsize': 6, 'xtick.labelsize': 5,
                         'ytick.labelsize': 5, 'axes.linewidth': 0.5,
                         'xtick.major.width': 0.5, 'ytick.major.width': 0.5,
                         'xtick.major.size': 2, 'ytick.major.size': 2}):
        fig, (ax_ridge, ax_mean) = plt.subplots(
            2, 1, figsize=(3, 3), dpi=300, sharex=True,
            gridspec_kw=dict(height_ratios=[3, 1.5], hspace=0.1,
                             left=0.16, right=0.95, top=0.84, bottom=0.14))

        ridge_plot(ridge, x, color=color, ax=ax_ridge, alpha=0.9,
                   line_width=0.5, trace_spacing=trace_spacing)
        for xv in (0, 1):
            ax_ridge.axvline(xv, color='k', linestyle='--', linewidth=0.5)
        ax_ridge.set_yticks([])
        ax_ridge.spines[['left', 'bottom']].set_visible(False)
        ax_ridge.tick_params(axis='x', length=0)
        plt.setp(ax_ridge.get_xticklabels(), visible=False)
        ax_ridge.set_ylabel('Cell', labelpad=2)

        x_sb = x[-1] + 0.02 * (x[-1] - x[0])
        ax_ridge.plot([x_sb, x_sb], [1, 1 + scale_bar], color='k', linewidth=0.8,
                      clip_on=False)
        ax_ridge.text(x_sb, 1 + scale_bar / 2, f' {scale_bar:g} z', va='center',
                      ha='left', fontsize=5)

        ax_mean.axvspan(0, 1, color=color, alpha=0.1, linewidth=0)
        ax_mean.plot(x, mean, color=color, linewidth=0.8)
        ax_mean.fill_between(x, mean - sem, mean + sem, color=color, alpha=0.3, linewidth=0)
        for xv in (0, 1):
            ax_mean.axvline(xv, color='k', linestyle='--', linewidth=0.5)
        ax_mean.axhline(0, color='gray', linewidth=0.3)
        ax_mean.set_xticks(xticks)
        ax_mean.set_xticklabels(xlabels)
        ax_mean.set_xlim(x[0], x[-1])
        ax_mean.set_xlabel(f'{category} bout (normalized)', labelpad=2)
        ax_mean.set_ylabel('z-score', labelpad=2)
        ax_mean.spines[['top', 'right']].set_visible(False)

        dir_lbl  = f' ({direction})' if direction else ''
        cell_lbl = (f'top {len(shown)} of {len(traces)} cells' if max_cells
                    else f'{len(traces)} cells')
        fig.suptitle(f'{category}{dir_lbl} — {session}\n'
                     f'{cell_lbl}, {sum(n_bouts.values())} bouts, '
                     f'{len(n_bouts)} animals, median {med:.1f}s',
                     fontsize=6, y=0.97)

        if save_path:
            fig.savefig(save_path, transparent=True)
        plt.show()

    return dict(x=x, traces=traces, labels=labels, n_bouts=n_bouts,
                durations=durations, order=order)

In [ ]:
res = plot_tuned_eta_warped_ridge('Aggressive', bout_len=.5, s_pre=2, session='R2',
                                  direction='excited', max_cells=30,
                                  save_path='figures/aggressive_eta_warped_ridge.svg')